# YOROSIX Cooperation Dataset - Social Simulation Example

This notebook demonstrates how to use the YOROSIX Cooperation Dataset for social simulation tasks.

## Use Cases

- Predict policy impact propagation
- Model organizational cascade dynamics
- Simulate market reaction patterns
- Multi-agent economic modeling

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import json

# Load the YOROSIX Cooperation Dataset
nodes = pd.read_csv('../data/ml_node_features_v6_3.csv')
edges = pd.read_csv('../data/ml_edge_list_v6_3.csv')
labels = pd.read_csv('../data/ml_node_label_dataset_v6_3.csv')
scores = pd.read_csv('../data/bdf_f1pple_score_v0_3.csv')

# Load temporal action sequences
with open('../data/bdc_action_sequence_v6-3.json', 'r') as f:
    actions = json.load(f)

print(f'Nodes: {len(nodes)}')
print(f'Edges: {len(edges)}')
print(f'Action sequences: {len(actions) if isinstance(actions, list) else "loaded"}')

## 2. Simulate Cooperation Cascade

Simulate how an idea or policy propagates through the network starting from an initial seed.

In [ ]:
# Build the network
G = nx.Graph()
for _, row in edges.iterrows():
    G.add_edge(int(row.iloc[0]), int(row.iloc[1]), 
               weight=float(row.iloc[2]) if len(row) > 2 else 1.0)

# Select high-influence seed nodes (top 5 by ripple score)
high_score = scores.sort_values('f1pple_score', ascending=False).head(5)
seed_nodes = high_score['member_id'].tolist()

print(f'Seed nodes: {seed_nodes}')
print(f'Their ripple scores: {high_score["f1pple_score"].tolist()}')

In [ ]:
# Simulate cascade propagation
def simulate_cascade(G, seeds, scores_df, threshold=0.5, max_steps=10):
    """Simulate cooperation cascade from seed nodes"""
    activated = set(seeds)
    history = [list(activated)]
    
    for step in range(max_steps):
        new_activations = set()
        
        # Check each non-activated node
        for node in G.nodes():
            if node in activated:
                continue
            
            # Count activated neighbors
            activated_neighbors = sum(1 for n in G.neighbors(node) if n in activated)
            total_neighbors = G.degree(node)
            
            if total_neighbors == 0:
                continue
            
            # Get node's cooperation propensity
            node_score = scores_df[scores_df['member_id'] == node]['f1pple_score'].values
            if len(node_score) == 0:
                continue
            
            # Probability of activation
            neighbor_ratio = activated_neighbors / total_neighbors
            activation_prob = neighbor_ratio * float(node_score[0])
            
            if activation_prob > threshold:
                new_activations.add(node)
        
        if not new_activations:
            break
        
        activated.update(new_activations)
        history.append(list(activated))
    
    return activated, history

final_activated, history = simulate_cascade(G, seed_nodes, scores, threshold=0.3)

print(f'Cascade complete after {len(history)} steps')
print(f'Total activated: {len(final_activated)} out of {G.number_of_nodes()} ({100*len(final_activated)/G.number_of_nodes():.1f}%)')
print(f'\nGrowth per step:')
for i, h in enumerate(history):
    print(f'  Step {i}: {len(h)} activated')

## 3. Visualize Cascade Propagation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)

# Show cascade at 3 time points
snapshots = [0, len(history)//2, len(history)-1]
titles = ['Initial (Seed Nodes)', 'Mid-Cascade', 'Final State']

for ax, snap_idx, title in zip(axes, snapshots, titles):
    activated = set(history[snap_idx])
    
    node_colors = ['red' if n in activated else 'lightblue' for n in G.nodes()]
    node_sizes = [100 if n in activated else 30 for n in G.nodes()]
    
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.05, width=0.3, ax=ax)
    
    ax.set_title(f'{title}\n({len(activated)} activated)', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Cascade Growth Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

steps = list(range(len(history)))
counts = [len(h) for h in history]

ax.plot(steps, counts, marker='o', linewidth=2, markersize=10, color='red')
ax.fill_between(steps, counts, alpha=0.3, color='red')

ax.set_xlabel('Cascade Step', fontsize=12)
ax.set_ylabel('Activated Nodes', fontsize=12)
ax.set_title('Cooperation Cascade Growth', fontsize=14)
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate(f'Seed: {counts[0]} nodes', xy=(0, counts[0]), 
            xytext=(0.5, counts[0]+5), fontsize=10)
ax.annotate(f'Final: {counts[-1]} nodes\n({100*counts[-1]/G.number_of_nodes():.1f}% coverage)', 
            xy=(steps[-1], counts[-1]), xytext=(steps[-1]-2, counts[-1]+15), fontsize=10)

plt.tight_layout()
plt.show()

## 5. Identify Critical Nodes

Find nodes that are most influential in cooperation propagation (highest betweenness).

In [ ]:
# Calculate betweenness centrality
betweenness = nx.betweenness_centrality(G, k=100, seed=42)

# Find top critical nodes
critical_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

print('Top 10 Critical Nodes (highest betweenness):')
print('These nodes are bridges in the network - removing them disrupts cooperation cascades.')
print()
for node_id, score in critical_nodes:
    coop_score = scores[scores['member_id'] == node_id]['f1pple_score'].values
    coop_str = f'{coop_score[0]:.4f}' if len(coop_score) > 0 else 'N/A'
    print(f'  Node {node_id}: betweenness={score:.4f}, cooperation_score={coop_str}')

## Use Cases for This Analysis

### Policy Simulation
- Test policy adoption by selecting different seed groups
- Compare cascade speed and coverage

### Organizational Resilience
- Identify single points of failure (critical nodes)
- Design redundant communication paths

### Marketing Strategy
- Find influencers for product launches
- Predict viral spread patterns

### Risk Management
- Model financial contagion
- Predict organizational collapse cascades

## Commercial Applications

Enterprise customers use this data for:
- Crisis prediction and management
- Market launch strategy
- Organizational design
- Policy impact assessment

**Contact**: kazutaka.hayashi@yorosix.com for commercial licensing